<a href="https://colab.research.google.com/github/Samse394/svm/blob/tutorial6/SVM_Sleep_Health.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

In [ ]:
df = pd.read_csv("Sleep_health_and_lifestyle_dataset.csv")

print("First five rows:")
print(df.head())

print("\nDataset shape:")
print(df.shape)

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
print(df.describe())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nTarget distribution (Sleep Disorder):")
print(df["Sleep Disorder"].fillna("None").value_counts())

In [ ]:
# Remove duplicate records
df = df.drop_duplicates()

# Prepare features and target for the Sleep Dataset
# We drop 'Person ID' because it's a unique identifier, and 'Sleep Disorder' which is the target
X = df.drop(["Person ID", "Sleep Disorder"], axis=1)
y = df["Sleep Disorder"].fillna("None")

# Identify numerical and categorical columns dynamically
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=["int64", "float64"]
).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessing pipeline created successfully for the Sleep Dataset.")

In [ ]:
# Perform the train-test split using the newly prepared Sleep Dataset variables
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Training targets:", y_train.unique())

In [ ]:
# Train and evaluate Linear SVM on Sleep Dataset
linear_svm = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", SVC(kernel="linear", C=1.0))
])

linear_svm.fit(X_train, y_train)
y_pred_linear = linear_svm.predict(X_test)

print("Linear SVM Accuracy:", accuracy_score(y_test, y_pred_linear))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_linear))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_linear))

In [ ]:
# Train and evaluate Non-Linear SVM on Sleep Dataset
nonlinear_svm = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", SVC(kernel="rbf", C=1.0, gamma="scale"))
])

nonlinear_svm.fit(X_train, y_train)
y_pred_rbf = nonlinear_svm.predict(X_test)

print("Non-Linear SVM (RBF) Accuracy:", accuracy_score(y_test, y_pred_rbf))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rbf))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rbf))

In [ ]:
# Compare the final SVM models dynamically on the Sleep Dataset
results = pd.DataFrame({
    "Model": ["Linear SVM", "Non-Linear SVM (RBF)"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_linear),
        accuracy_score(y_test, y_pred_rbf)
    ]
})

print("Model Comparison Results:")
print(results)

# Plot the comparison bar chart
results.plot(
    x="Model",
    y="Accuracy",
    kind="bar",
    legend=False,
    ylim=(0, 1.1),
    title="Comparison of Linear and Non-Linear SVM (Sleep Dataset)",
    color=["#1f77b4", "#ff7f0e"]
)

plt.ylabel("Accuracy")
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Create RBF SVM model
rbf_svm = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", SVC(kernel="rbf", C=1.0, gamma="scale"))
])

# Train model
rbf_svm.fit(X_train, y_train)

# Make predictions
y_pred_rbf = rbf_svm.predict(X_test)

# Evaluate model
print("RBF SVM Accuracy:",
      accuracy_score(y_test, y_pred_rbf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rbf, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rbf))

In [ ]:
# Create Sigmoid SVM model
sigmoid_svm = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", SVC(kernel="sigmoid", C=1.0, gamma="scale"))
])

# Train model
sigmoid_svm.fit(X_train, y_train)

# Make predictions
y_pred_sigmoid = sigmoid_svm.predict(X_test)

# Evaluate model
print("Sigmoid SVM Accuracy:",
      accuracy_score(y_test, y_pred_sigmoid))

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred_sigmoid, zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_sigmoid))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results = pd.DataFrame({
    "Kernel": ["RBF", "Sigmoid"],
    "Accuracy (%)": [
        accuracy_score(y_test, y_pred_rbf) * 100,
        accuracy_score(y_test, y_pred_sigmoid) * 100
    ]
})

print(results.round(2))

plt.figure(figsize=(7, 4))
plt.bar(results["Kernel"], results["Accuracy (%)"])
plt.xlabel("SVM Kernel")
plt.ylabel("Accuracy (%)")
plt.title("RBF vs Sigmoid SVM")
plt.ylim(0, 100)
plt.show()